## Compare models

### Basic Parameter

Note: Do not rename any of the variables in this section, because the they are externally used in github action `Train Model`

In [ ]:
Input_Dir  = 'data_raw_all'
Output_Dir = 'models'
Save_Evaluation_Images = False

# Input image size [x, y, channels]
Input_Shape = (32, 32, 3)

# Validation size
# Note: 0.0 = 0% validation size, use all images for training
Validation_Percentage = 0.2


### Load libraries and defaults

In [ ]:
import os
import glob
from pathlib import Path
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tensorflow import keras
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

from src.utils.plot_functions import plot_dataset_analog_result, plot_divergence


%matplotlib inline
np.set_printoptions(precision=4)
np.set_printoptions(suppress=True)


### Load files

In [ ]:
files = glob.glob(Input_Dir + '/*.jpg')
y_data = np.empty((len(files)))
f_data = np.empty((len(files)), dtype="<U250")
x_data = np.empty((len(files),Input_Shape[0],Input_Shape[1],Input_Shape[2]))

for i, aktfile in enumerate(files):
    base = os.path.basename(aktfile)

    # get label from filename (1.2_ new or 1_ old),
    if (base[1]=="."):
        target = base[0:3]
    else:
        target = base[0:1]
    category = float(target)

    image_bytes = tf.io.read_file(aktfile)
    image = tf.image.decode_image(image_bytes, channels=Input_Shape[2], expand_animations=False)
    image = tf.image.resize(image, [Input_Shape[0],Input_Shape[1]], method=tf.image.ResizeMethod.MITCHELLCUBIC)
    image = tf.cast(image, tf.float32)
    image = np.array(image, dtype="float32")
    
    f_data[i] =  aktfile
    x_data[i] = image
    y_data[i] =  category

print("Data count: ", len(y_data))  
print(x_data.shape)

x_data, y_data, f_data = shuffle(x_data, y_data, f_data)
X_train, X_test, y_train, y_test = train_test_split(x_data, y_data, test_size=Validation_Percentage)
y_train = keras.utils.to_categorical(y_train*10, 100)
y_test = keras.utils.to_categorical(y_test*10, 100)

print(np.expand_dims(y_data, axis=1).shape)


### Helper functions

In [ ]:
def plot_dataset_analog_result_sorted(false_predicted_images, false_predicted_plt_labels, false_predicted_dev_values):
    # Sort by deviation (largest first)
    sorted_items = sorted(
        zip(false_predicted_images, false_predicted_plt_labels, false_predicted_dev_values),
        key=lambda item: item[2],  # Sort by false_predicted_dev_values
        reverse=True
    )
    
    false_predicted_images, false_predicted_plt_labels, false_predicted_dev_values = zip(*sorted_items)
    false_predicted_images = list(false_predicted_images)
    false_predicted_plt_labels = list(false_predicted_plt_labels)
    false_predicted_dev_values = list(false_predicted_dev_values)
    
    # Plot results
    print("False Predictions (Sorted by largest deviation, max. 49 images)")
    plot_dataset_analog_result(false_predicted_images, false_predicted_plt_labels, columns=7, rows=7, figsize=(18,18))


def evaluate_model_accuracy(model_path, x_data, y_data, f_data, title, accepted_deviation = 0.1, save_images=False):
    false_predicted_images = []
    false_predicted_plt_labels = []
    false_predicted_dev_values = []

    # Prepare interpreter and load model
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    input_index = interpreter.get_input_details()[0]["index"]
    output_index = interpreter.get_output_details()[0]["index"]
    
    # Ignore models with other than defined shape (default: 32, 32, 3)
    if ((interpreter.get_input_details()[0]["shape"] != (1, Input_Shape[0], Input_Shape[1], Input_Shape[2])).any()):
        print("Input dimension not supported")
        return

    for x, y, f in zip(x_data, y_data, f_data):
        
        interpreter.set_tensor(input_index, np.expand_dims(x.astype(np.float32), axis=0))
        interpreter.invoke() # Run inference
        
        # Post-processing: Remove batch dimension and find the result with highest probability
        output = interpreter.get_tensor(output_index)

        # Calculate prediction value depending on model type (ana-cont, ana-class100)
        if (len(output[0])==2): # ana-cont
            out_sin = output[0][0]  
            out_cos = output[0][1]
            predicted_val = np.round(((np.arctan2(out_sin, out_cos) / (2 * math.pi)) % 1) * 10, 1)
        elif (len(output[0])==100): # ana-class100
            predicted_val = (np.argmax(output, axis=1).reshape(-1) / 10)[0]
        else:
            print("Output dimension not supported")
            return

        deviation_val = np.round(min(abs(predicted_val - y), abs(predicted_val - (10 - y))) * 10) / 10

        # Round values
        predicted_val = np.round(predicted_val, 1)
        expected_val = np.round(y, 1)
        deviation_val = np.round(deviation_val, 1)
        
        if deviation_val > accepted_deviation:
            false_predicted_images.append(x)
            false_predicted_plt_labels.append(
                "Expected: " + str(expected_val) + 
                "\n Predicted: " + str(predicted_val) + 
                "\n" + str(os.path.basename(f)[:-4][:20])
            )
            false_predicted_dev_values.append(deviation_val)
            #print(predicted_val, y, deviation_val)
    
    accuracy = "{:.2f}%".format((1-len(false_predicted_dev_values)/len(y_data))*100)
    title = f"Model: {os.path.basename(model_path)}  |  Images: {len(y_data)}\nAccuracy: {accuracy} (False Predicted: {len(false_predicted_dev_values)}) With Accepted Deviation: {accepted_deviation}"
    
    if save_images:
        filename = os.path.join(Output_Dir, os.path.basename(model_path) + ".png")
    else:
        filename = None

    
    # Plot the deviation divergence (max deviation: 5.0)
    plot_divergence(np.bincount(np.array(np.array(false_predicted_dev_values) * 10).astype(int), minlength=51), title, filename)


    # Plot the dataset of false predictions (Use first 49 entries, sorted by deviation (largest first)
    # plot_dataset_analog_result_sorted(false_predicted_images, false_predicted_plt_labels, false_predicted_dev_values)


### Evaluate Model Accuracy (Accepted Deviation: 0.1)

In [ ]:
modelfiles = sorted(glob.glob('models/ana-class100/*.tflite')) + sorted(glob.glob('models/ana-cont/*.tflite')) + sorted(glob.glob('models_tmp/*.tflite'))

for modelfile in modelfiles:
    evaluate_model_accuracy(modelfile, x_data, y_data, f_data, title=modelfile, accepted_deviation=0.1, save_images=Save_Evaluation_Images)
    print(f" ")
    print(f" ")
    

### Evaluate Model Accuracy (Accepted Deviation: 0.0)

In [ ]:
for modelfile in modelfiles:
    evaluate_model_accuracy(modelfile, x_data, y_data, f_data, title=modelfile, accepted_deviation=0.0)
    print(f" ")
    print(f" ")
